# Cross-Source PropAMM Quote Ladder Analysis

Polaris normalizes on-chain proprietary AMM observations into one
[PropAMM Quote Ladders](https://docs.polaris.supply/schemas/propamm-quote-ladders) schema. This
notebook checks all six documented sources, selects a directed token pair shared by the most sources,
and compares observed quote curves and price impact without losing uint256 precision. Recorded
ladder bounds are treated as probe ranges, not as evidence of executable liquidity.


## Setup

Quote ladders are sparse block-level events. Each source is queried over up to the first hour of
its no-key preview day, with a 5,000-row cap.


In [ ]:
from itertools import islice, product
import warnings

from polaris_data import PolarisClient
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
warnings.filterwarnings("ignore", message=r".*snapshot coverage.*", category=UserWarning)


def as_utc(value):
    timestamp = pd.Timestamp(value)
    return timestamp.tz_localize("UTC") if timestamp.tzinfo is None else timestamp.tz_convert("UTC")


def accessible_bounds(market_info):
    """Return the no-key catalog interval for an open or preview market."""
    start = as_utc(market_info["start"])
    end = as_utc(market_info["end"])
    access = market_info.get("access") or {}
    cutoff = access.get("public_cutoff_date")
    if access.get("status") == "preview" and cutoff:
        public_day = as_utc(cutoff)
        start = max(start, public_day)
        end = min(end, public_day + pd.Timedelta(days=1))
    if start >= end:
        raise ValueError("Catalog metadata does not expose a no-key interval for this market")
    return start, end


def bounded_rows(iterator, limit):
    """Materialize at most limit rows and close a partially consumed SDK generator."""
    rows = list(islice(iterator, limit + 1))
    truncated = len(rows) > limit
    close = getattr(iterator, "close", None)
    if close is not None:
        close()
    return rows[:limit], truncated


def event_timestamp(row):
    """Support both the legacy and v2 Polaris event envelopes."""
    value = row.get("collector_timestamp", row.get("timestamp"))
    return pd.to_datetime(value, unit="ms", utc=True)

from decimal import Decimal


In [ ]:
sources = ["fermiswap", "bopamm", "kipseli", "metric", "tempest", "taurusfi"]
market = "ethereum"
window_length = pd.Timedelta(hours=1)
max_rows_per_source = 5_000
max_block_skew = 12
max_time_skew = pd.Timedelta(minutes=2)
baseline_input_target = Decimal("0.01")
representative_inputs = [Decimal("0.01"), Decimal("1"), Decimal("10")]


## Discover and fetch each source

Empty sources remain in the coverage table so absence in a short public window is not confused with
an unsupported schema.


In [ ]:
rows_by_source = {}
coverage_records = []

with PolarisClient() as client:
    for source in sources:
        catalog = client.catalog(source=source, market=market)
        if not catalog.get("markets"):
            rows_by_source[source] = []
            coverage_records.append({"source": source, "rows": 0, "hit_cap": False, "window": "not in catalog"})
            continue

        market_info = catalog["markets"][0]
        start, accessible_end = accessible_bounds(market_info)
        end = min(start + window_length, accessible_end)
        rows, truncated = bounded_rows(
            client.propamm_quote_ladders(
                source=source, market=market, from_=start, to=end, allow_gaps=True,
            ),
            max_rows_per_source,
        )
        rows_by_source[source] = rows
        coverage_records.append({
            "source": source,
            "rows": len(rows),
            "hit_cap": truncated,
            "window": f"{start} -> {end}",
        })

coverage_df = pd.DataFrame(coverage_records)
coverage_df


## Inspect event identity and select a comparable pair

Comparison is restricted to the same token direction. A pair seen across the most sources wins;
ties prefer the pair with the most quote observations. If only one source is populated, its richest
pair still produces a complete single-source analysis.


In [ ]:
ladder_records = []
for source, rows in rows_by_source.items():
    for row in rows:
        values = (row.get("data") or {}).get("values") or {}
        ladder_records.append({
            "source": source,
            "participant": (
                source if not values.get("pool") else f"{source}:{values['pool'].lower()}"
            ),
            "timestamp": event_timestamp(row),
            "collector_sequence": row.get("collector_sequence"),
            "chain_id": values.get("chain_id"),
            "token_in": (values.get("token_in") or "").lower(),
            "token_out": (values.get("token_out") or "").lower(),
            "token_in_decimals": values.get("token_in_decimals"),
            "token_out_decimals": values.get("token_out_decimals"),
            "quotes": values.get("quotes") or [],
            "event_id": values.get("event_id"),
            "block_number": values.get("block_number"),
            "block_hash": values.get("block_hash"),
            "parent_hash": values.get("parent_hash"),
            "transaction_hash": values.get("transaction_hash"),
            "transaction_index": values.get("transaction_index"),
            "router": values.get("router"),
            "oracle": values.get("oracle"),
            "pool": values.get("pool"),
        })

ladders_df = pd.DataFrame(ladder_records)
if ladders_df.empty:
    raise ValueError("No PropAMM quote ladders were found in the public sample windows")

pair_coverage = (
    ladders_df.groupby(["token_in", "token_out"])
    .agg(sources=("source", "nunique"), ladders=("source", "size"), quote_points=("quotes", lambda values: sum(map(len, values))))
    .sort_values(["sources", "quote_points", "ladders"], ascending=False)
)
selected_pair = pair_coverage.index[0]
print(f"Selected directed pair: {selected_pair[0]} -> {selected_pair[1]}")
pair_coverage.head(10)


## Align ladders and compare exact shared input sizes

The least frequently observed participant is used as the cohort anchor. For each of its ladders,
the nearest ladder from every other participant is retained only when the whole cohort fits the
configured block and observation-time skew. Metric pools remain separate participants.

Amounts stay as `Decimal`. Comparisons use exact input amounts shared by every participant and a
common economically meaningful baseline. This avoids using tiny, output-quantized quotes as the
price-impact reference. `recorded_max_probe` is only the largest sampled input—not capacity.


In [ ]:
def align_nearest_ladders(pair_ladders):
    """Build unique, complete nearest-block cohorts within explicit skew limits."""
    pair_ladders = pair_ladders.dropna(subset=["block_number", "timestamp"]).copy()
    participant_counts = pair_ladders.groupby("participant").size()
    participants = sorted(participant_counts.index)
    minimum_count = participant_counts.min()
    anchor_participant = sorted(participant_counts[participant_counts.eq(minimum_count)].index)[0]
    aligned_records = []
    cohort_records = []
    seen_event_sets = set()

    for _, anchor in pair_ladders[pair_ladders["participant"].eq(anchor_participant)].iterrows():
        candidate_groups = []
        for participant in participants:
            if participant == anchor_participant:
                candidate_groups.append([anchor])
                continue
            candidates = pair_ladders[pair_ladders["participant"].eq(participant)]
            candidates = candidates[
                (candidates["block_number"] - anchor["block_number"]).abs().le(max_block_skew)
                & (candidates["timestamp"] - anchor["timestamp"]).abs().le(max_time_skew)
            ]
            candidate_groups.append([candidate for _, candidate in candidates.iterrows()])

        if any(not candidates for candidates in candidate_groups):
            continue
        valid_combinations = []
        for combination in product(*candidate_groups):
            block_span = int(max(member["block_number"] for member in combination) - min(member["block_number"] for member in combination))
            time_span = max(member["timestamp"] for member in combination) - min(member["timestamp"] for member in combination)
            if block_span <= max_block_skew and time_span <= max_time_skew:
                total_block_distance = sum(abs(member["block_number"] - anchor["block_number"]) for member in combination)
                total_time_distance = sum((abs(member["timestamp"] - anchor["timestamp"]) for member in combination), pd.Timedelta(0))
                valid_combinations.append((block_span, time_span, total_block_distance, total_time_distance, combination))
        if not valid_combinations:
            continue
        _, time_span, _, _, members = min(valid_combinations, key=lambda candidate: candidate[:4])
        block_span = int(max(member["block_number"] for member in members) - min(member["block_number"] for member in members))

        event_set = tuple(sorted(member["event_id"] for member in members))
        if event_set in seen_event_sets:
            continue
        seen_event_sets.add(event_set)
        cohort_id = len(cohort_records)

        cohort_records.append({
            "cohort_id": cohort_id,
            "anchor_participant": anchor_participant,
            "anchor_block": int(anchor["block_number"]),
            "anchor_timestamp": anchor["timestamp"],
            "block_span": block_span,
            "time_span_seconds": time_span.total_seconds(),
        })
        for member in members:
            record = member.drop(labels=["_block_distance", "_time_distance"], errors="ignore").to_dict()
            record.update({
                "cohort_id": cohort_id,
                "block_offset": int(member["block_number"] - anchor["block_number"]),
                "time_offset_seconds": (member["timestamp"] - anchor["timestamp"]).total_seconds(),
            })
            aligned_records.append(record)

    if not cohort_records:
        raise ValueError("No complete quote-ladder cohorts satisfy the configured skew limits")
    return pd.DataFrame(aligned_records), pd.DataFrame(cohort_records)


def compare_exact_shared_inputs(cohort_ladders):
    """Normalize one cohort and calculate level and curve metrics on exact shared sizes."""
    quote_records = []
    probe_records = []
    for _, ladder in cohort_ladders.iterrows():
        input_scale = Decimal(10) ** int(ladder["token_in_decimals"])
        output_scale = Decimal(10) ** int(ladder["token_out_decimals"])
        participant_quotes = []
        for quote in ladder["quotes"]:
            amount_in = Decimal(quote["amount_in"]) / input_scale
            amount_out = Decimal(quote["amount_out"]) / output_scale
            if amount_in <= 0:
                continue
            participant_quotes.append((amount_in, amount_out))
            quote_records.append({
                "cohort_id": ladder["cohort_id"],
                "source": ladder["source"],
                "participant": ladder["participant"],
                "amount_in": amount_in,
                "amount_out": amount_out,
                "average_rate": amount_out / amount_in,
            })
        if not participant_quotes:
            raise ValueError(f"{ladder['participant']} has no positive quote inputs")
        probe_records.append({
            "cohort_id": ladder["cohort_id"],
            "source": ladder["source"],
            "participant": ladder["participant"],
            "recorded_quote_points": len(participant_quotes),
            "recorded_min_probe": min(amount for amount, _ in participant_quotes),
            "recorded_max_probe": max(amount for amount, _ in participant_quotes),
        })

    normalized = pd.DataFrame(quote_records)
    input_sets = [set(values["amount_in"]) for _, values in normalized.groupby("participant")]
    exact_shared_inputs = sorted(set.intersection(*input_sets))
    eligible_inputs = [amount for amount in exact_shared_inputs if amount >= baseline_input_target]
    if not eligible_inputs:
        raise ValueError("No exact shared input meets the configured baseline target")

    baseline_input = eligible_inputs[0]
    compared = normalized[normalized["amount_in"].isin(eligible_inputs)].copy()
    baseline_rates = (
        compared[compared["amount_in"].eq(baseline_input)]
        .set_index("participant")["average_rate"]
    )
    best_rates = compared.groupby("amount_in")["average_rate"].max()
    compared["baseline_rate"] = compared["participant"].map(baseline_rates)
    compared["best_rate"] = compared["amount_in"].map(best_rates)
    compared["quote_edge_bps"] = compared.apply(
        lambda row: (row["average_rate"] / row["best_rate"] - Decimal(1)) * Decimal(10_000), axis=1
    )
    compared["curve_impact_bps"] = compared.apply(
        lambda row: (row["average_rate"] / row["baseline_rate"] - Decimal(1)) * Decimal(10_000), axis=1
    )
    for column in ["amount_in", "amount_out", "average_rate", "quote_edge_bps", "curve_impact_bps"]:
        compared[f"{column}_float"] = compared[column].astype(float)

    details = {
        "exact_shared_points": len(exact_shared_inputs),
        "shared_min_input": exact_shared_inputs[0],
        "shared_max_input": exact_shared_inputs[-1],
        "baseline_input": baseline_input,
        "analysis_points": len(eligible_inputs),
    }
    return compared, pd.DataFrame(probe_records), details


pair_ladders = ladders_df[
    ladders_df["token_in"].eq(selected_pair[0]) & ladders_df["token_out"].eq(selected_pair[1])
].copy()
aligned_ladders, cohort_summary = align_nearest_ladders(pair_ladders)

comparison_frames = []
probe_frames = []
comparison_details = []
for cohort_id, cohort_ladders in aligned_ladders.groupby("cohort_id"):
    compared, probes, details = compare_exact_shared_inputs(cohort_ladders)
    comparison_frames.append(compared)
    probe_frames.append(probes)
    comparison_details.append({"cohort_id": cohort_id, **details})

all_comparisons = pd.concat(comparison_frames, ignore_index=True)
all_probe_ranges = pd.concat(probe_frames, ignore_index=True)
cohort_summary = cohort_summary.merge(pd.DataFrame(comparison_details), on="cohort_id")
selected_cohort_id = cohort_summary.sort_values("anchor_timestamp").iloc[-1]["cohort_id"]
selected_ladders = aligned_ladders[aligned_ladders["cohort_id"].eq(selected_cohort_id)].copy()
quotes_df = all_comparisons[all_comparisons["cohort_id"].eq(selected_cohort_id)].copy()
probe_summary = all_probe_ranges[all_probe_ranges["cohort_id"].eq(selected_cohort_id)].copy()

metadata_columns = [
    "participant", "timestamp", "block_number", "block_offset", "time_offset_seconds",
    "transaction_hash", "router", "oracle", "pool", "event_id",
]
display(cohort_summary)
display(selected_ladders[metadata_columns].sort_values("participant"))
display(probe_summary.drop(columns="cohort_id").set_index("participant"))

available_representative_inputs = sorted(
    set(amount for amount in representative_inputs if amount in set(quotes_df["amount_in"]))
    | {quotes_df["amount_in"].max()}
)
representative = quotes_df[quotes_df["amount_in"].isin(available_representative_inputs)]
representative_comparison = pd.concat({
    "amount_out": representative.pivot(index="amount_in", columns="participant", values="amount_out_float"),
    "quote_edge_bps": representative.pivot(index="amount_in", columns="participant", values="quote_edge_bps_float"),
}, axis=1)
display(representative_comparison)

cross_cohort_summary = all_comparisons.groupby("participant").agg(
    aligned_observations=("cohort_id", "nunique"),
    shared_quote_points=("amount_in", "size"),
    quote_point_win_rate=("quote_edge_bps_float", lambda values: values.abs().lt(1e-9).mean()),
    median_quote_edge_bps=("quote_edge_bps_float", "median"),
    p05_quote_edge_bps=("quote_edge_bps_float", lambda values: values.quantile(0.05)),
    median_curve_impact_bps=("curve_impact_bps_float", "median"),
    worst_curve_impact_bps=("curve_impact_bps_float", "min"),
)
cross_cohort_summary


## Shared-size quote level and price impact

The latest valid cohort is plotted only on exact shared inputs at or above the common baseline.
Quote edge measures each participant against the best observed output at the same input size; curve
impact measures its rate change from the common baseline. A quote remains an observation, not proof
that the same amount can be settled later; capacity requires venue-aware transaction simulation or
settlement evidence.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for source, values in quotes_df.groupby("participant"):
    values = values.sort_values("amount_in_float")
    axes[0].plot(values["amount_in_float"], values["average_rate_float"], marker=".", label=source)
    axes[1].plot(values["amount_in_float"], values["quote_edge_bps_float"], marker=".", label=source)
    axes[2].plot(values["amount_in_float"], values["curve_impact_bps_float"], marker=".", label=source)

for axis in axes:
    axis.set_xscale("log")
    axis.set_xlabel("Input token amount")
    axis.legend()
axes[0].set_title("Average output per input")
axes[0].set_ylabel("Output per input")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Quote edge vs best at same size")
axes[1].set_ylabel("Edge (bps)")
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set_title(f"Curve impact vs {quotes_df['amount_in'].min()} input")
axes[2].set_ylabel("Impact (bps)")

plt.tight_layout()
plt.show()
